# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises can help alleviate lower back discomfort and improve flexibility and strength in tha

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. Adequate sleep, typically 7-9 hours per night, helps the body repair tissues, consolidate memories, and regulate hormones related to growth and appetite. Good sleep quality and hygiene, such as maintaining a consistent sleep schedule, creating a restful environment, and establishing relaxing routines, can improve immune function, reduce stress, and enhance mood. Conversely, poor sleep or sleep disorders like insomnia can negatively impact physical health, mental health, and daily functioning.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. For headaches, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, and using essential oils such as peppermint or lavender.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Complete 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds. Repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. Maintaining a regular sleep schedule and creating an optimal sleep environment—such as keeping the room cool, dark, and quiet—can improve sleep quality. Good sleep hygiene practices, like minimizing screen time before bed, avoiding caffeine in the late afternoon, and establishing relaxing bedtime routines, support restful sleep. Quality sleep helps restore the body, supports immune function, enhances mental health, and promotes physical recovery, all contributing to overall wellness.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may also help promote relaxation and reduce headache symptoms. Additionally, staying well-hydrated, ensuring adequate sleep, and managing stress through activities like mindfulness can help alleviate headaches related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

BM25 is better than embeddings when the query relies on exact keyword matching, such as searching for specific error codes, unique product IDs, or acronyms (e.g., "Error 404 Not Found", "iPhone 17 Pro Max", or "NATO").

This happens because BM25 is a sparse retrieval method based on exact term frequencies (keyword matching). It excels when the user expects to find documents containing the exact words they searched for. Dense embeddings, which retrieve based on semantic meaning, might return conceptually similar results but completely miss the exact identifier or keyword the user actually needed.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'According to the provided information, exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on your hands and knees. Arch your back upward (like a cat) and then let it sag downward (like a cow). Repeat 10-15 times.\n\n2. **Bird Dog:** From hands and knees, extend opposite arm and leg simultaneously while keeping your core engaged. Hold the position for 5 seconds, then switch sides. Do 10 repetitions on each side.\n\n3. **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep positively affects overall health by supporting physical repair, memory consolidation, and hormone regulation. Adequate sleep, typically 7-9 hours per night, helps the body repair tissues, regulates growth and appetite hormones, and maintains mental well-being and cognitive function. The quality and environment of sleep are also important factors for promoting good health.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n- Drinking water to stay hydrated\n- Applying a cold or warm compress to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\nAdditionally, for immediate stress relief, deep breathing, progressive muscle relaxation, grounding techniques, taking a short walk, and listening to calming music can be helpful.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for about 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and focus on stretching and strengthening the lo

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, the body repairs tissues, regenerates cells, and delivers hormones essential for growth and appetite regulation. It helps consolidate memories, maintain mental well-being, and regulate emotions. Adequate sleep (typically 7-9 hours for adults) is also linked to a stronger immune system, better stress management, and a lower risk of chronic conditions such as headaches, digestive issues, and mood disorders. Conversely, poor sleep or sleep disturbances can negatively impact health, leading to symptoms like fatigue, headaches, and increased vulnerability to illness. Therefore, prioritizing good sleep hygiene and creating an optimal sleep environment are important for maintaining overall health and wellness.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhaling for 4 counts, holding, exhaling, and holding again)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, such as naming things you see, hear, feel, smell, and taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n\nThese approaches can help provide immediate relief and contribute to managing stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because it anticipates different ways on how the user’s intent could be phrased. It retrieves documents for each distinct variation of the query and combines the unique results.
A single query might miss relevant documents that use different vocabulary, synonyms, or sentence structures (e.g., "lower back pain" vs. "lumbar discomfort"). By reformulating the query from multiple angles, the retrieval system casts a wider net, pulling in a broader set of relevant context that might have been skipped otherwise.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some recommended exercises include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate between arching your back upward (cat) and letting it sag downward (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg simultaneously, hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat, hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis upward to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle movements can help alleviate discomfort and prevent future episodes of lower back pain.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in several important ways. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body performs essential repair processes, such as tissue repair and regeneration. Sleep also helps consolidate memories and supports brain functions. Additionally, sleep influences the release of hormones that regulate growth and appetite, impacting metabolic health and weight management. Adults generally need 7-9 hours of quality sleep per night, which occurs in cycles involving REM and non-REM stages. Maintaining good sleep hygiene and creating an optimal sleep environment can enhance sleep quality, further promoting overall health and wellness.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in mindfulness or meditation, performing gentle stretching or yoga, taking a warm bath, listening to calming music, and using essential oils such as peppermint or lavender. Additionally, staying well-hydrated by drinking water, resting in a dark and quiet room, and getting regular, quality sleep can help manage these issues naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle stretching and strengthening movements designed to alleviate discomfort and prevent future episo

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical well-being, mental health, and cognitive functions. During sleep, the body engages in tissue repair, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night—supports the immune system, reduces stress, and helps maintain proper metabolic and hormonal balance. Poor sleep quality or insufficient sleep is linked to various health issues, including increased risk of chronic conditions such as heart disease, diabetes, weakened immunity, mental health problems like anxiety and depression, and impaired cognitive functioning. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are crucial for sustaining overall health and wellness.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For headaches:\n  - Drink water to stay hydrated\n  - Apply a cold or warm compress to the head or neck\n  - Rest in a dark, quiet room\n  - Gentle massage of temples and neck\n  - Use peppermint or lavender essential oils\n  - Maintain a regular sleep schedule\n  - In small amounts, caffeine can help or hurt depending on the individual\n\n- For stress relief:\n  - Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n  - Progressive muscle relaxation, tensing and releasing muscle groups\n  - Grounding techniques (naming sensory experiences)\n  - Taking short walks in nature\n  - Listening to calming music\n\nThese methods can help manage stress and headaches naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch**: Start on hands and knees; alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch**: Lie on your back; pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts**: Lie on your back with knees bent; flatten your back against the floor by tightening your abs and tilting the pelvis up slightly. Hold for 10 seconds; repeat 8-12 times.\n\n- **Bird Dog**: From hands and knees, extend the opposite arm and leg while keeping your core engaged for about 5 seconds, then switch sides.\n\nThese exercises are gentle stretching and strengthening movements designed to improve flexibility an

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. According to the provided information, sleep is essential for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7 to 9 hours of sleep per night, and quality sleep occurs in cycles of about 90 minutes, alternating between REM and non-REM stages. Good sleep hygiene practices—such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, keeping the bedroom cool and dark, and limiting screen exposure before bed—help improve sleep quality. Proper sleep supports immune function, reduces stress, enhances mood, optimizes cognitive performance, and helps prevent health issues like insomnia and other sleep disorders.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, mindfulness and meditation practices, grounding techniques (such as naming things you see, hear, feel, smell, and taste), spending time in nature, and engaging in hobbies or leisure activities. \n\nFor headaches, natural remedies include staying well-hydrated by drinking plenty of water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If sentences are short and repetitive, semantic chunking will likely group too many sentences together into excessively large chunks, or fail to find meaningful split points, because the semantic meaning between the repetitive sentences remains very similar.
To adjust the algorithm, we should lower the distance threshold (percentile threshold) that triggers a split. Making the threshold stricter forces the algorithm to be more sensitive to smaller changes in meaning, allowing it to appropriately split short, repetitive text. Alternatively, we could switch to a simple recursive character splitter for highly structured FAQ content.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
import time
import pandas as pd
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, ContextPrecision
from ragas import evaluate, EvaluationDataset

# 1. Synthetic Data Generation (SDG)
print("Generating Synthetic Test Data with Ragas...")
# We wrap the chat_model and embeddings from the previous tasks
generator_llm = LangchainLLMWrapper(chat_model)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# Generate a small testset for evaluation using the raw_docs from Task 2
testset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)
test_df = testset.to_pandas()

# Convert reference_contexts to strings for Ragas evaluation compatibility
test_df["reference_contexts"] = test_df["reference_contexts"].apply(lambda x: [str(c) for c in x])
print(f"Generated {len(test_df)} test cases.")

# 2. Setup Retrievers Dictionary
retrievers = {
    "Naive": naive_retriever,
    "BM25": bm25_retriever,
    "Multi-Query": multi_query_retriever,
    "Parent-Doc": parent_document_retriever,
    "Reranked": compression_retriever,
    "Ensemble": ensemble_retriever,
    "Semantic-Chunking": semantic_retriever
}

# 3. Benchmarking Loop
results = []

print("\nStarting Benchmark...")
for name, retriever in retrievers.items():
    start_time = time.time()
    retrieved_data = []
    
    print(f"Evaluating: {name}...")
    for _, row in test_df.iterrows():
        question = row["user_input"]
        
        # Fetch contexts using the retriever
        retrieved_docs = retriever.invoke(question)
        retrieved_contexts = [d.page_content for d in retrieved_docs]
        
        retrieved_data.append({
            "user_input": question,
            "retrieved_contexts": retrieved_contexts,
            "reference": row["reference"]
        })
        
        # Hard delay to ensure we definitively stay under 10 requests / 60 seconds
        time.sleep(6.5)
    
    # Calculate Latency (subtract artificial sleep time)
    latency = ((time.time() - start_time) - (6.5 * len(test_df))) / len(test_df)
    
    # Evaluate with Ragas
    eval_ds = EvaluationDataset.from_list(retrieved_data)
    eval_result = evaluate(
        dataset=eval_ds,
        metrics=[LLMContextRecall(), ContextPrecision()],
        llm=generator_llm
    )
    
    # Store results
    results.append({
        "Retriever": name,
        "Context Recall": eval_result["context_recall"],
        "Context Precision": eval_result["context_precision"],
        "Avg Latency (s)": round(latency, 3)
    })

# 4. Display Results
results_df = pd.DataFrame(results)
print("\nFinal Results:")
print(results_df.to_markdown(index=False))


Generating Synthetic Test Data with Ragas...


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/3 [00:00<?, ?it/s]

...

Evaluating: Ensemble...


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

Evaluating: Semantic-Chunking...


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]


Final Results:
| Retriever         | Context Recall                                                              | Context Precision                                                                                                                                                                                                              |   Avg Latency (s) |
|:------------------|:----------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------:|
| Naive             | [1.0, 1.0, 0.75, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]               | [0.5798611110966146, 0.9861111110987848, 0.0, 0.09999999999, 0.8666666666377778, 0.8755456349096906, 0.0, 0.7916666666402777, 0.9999999999, 0.6666666666333333, 0.9999999999, 0.49999999995]      

1. **Cost:** **BM25**, **Naive**, and **Semantic-Chunking** are the most cost-effective methods, requiring no external LLM overhead or reranking logic during the retrieval step. Conversely, **Multi-Query** and **Reranked** incur higher computational and LLM token costs per query to rewrite or prune documents.

2. **Latency:** **BM25** remains fundamentally the fastest (0.007s), followed tightly by **Semantic-Chunking** (0.167s), **Naive** (0.184s), and **Parent-Doc** (0.186s). Due to executing multiple concurrent retrievals, the **Ensemble** strategy takes the longest (2.108s), and **Multi-Query** is similarly sluggish (1.379s). 

3. **Performance:** While the **Ensemble** method improves recall to a perfect 1.00, it substantially drops precision (0.477) by casting too wide of a net over the text. The standout technique for this dataset is **Semantic-Chunking**. Because the wellness guide contains clean semantic breaks (categories on sleep, headers on back pain, etc.), Semantic-Chunking achieves a perfect Context Recall (1.00) and the highest Context Precision (0.852) among standalone retrievers.

**Conclusion:**
For this particular wellness data, the **Semantic-Chunking** strategy offers the best ROI. It consistently delivers flawless recall (1.00) while scoring higher overall precision (0.852) than the other combined search models natively. It effectively achieves higher accuracy by keeping highly related contextual sentences clumped together prior to the embedding step, practically neutralizing the need for complex dual-retrieval architectures (**Parent-Doc**) or slower, costlier pipelines (**Ensemble/Multi-Query**).